In [35]:
import numpy as np
import pandas as pd
import plotly.express as px
from glob import glob
import re

LAYER = 2

feats = glob(f"../../experiment_data/multiseed_matching/a{LAYER}_valleys/feats_*.csv")
points = glob(f"../../experiment_data/multiseed_matching/a{LAYER}_valleys/points_*.csv")

valley_seed_re = r"v(\d+)_.*_([abc]).csv"

def get_valley_seed(filename):
    match = re.search(valley_seed_re, filename)
    if match:
        valley_count = int(match.group(1))
        seed = match.group(2)
        return valley_count, seed
    else:
        return None, None
    
dps = list(map(lambda f: (*get_valley_seed(f), f), feats))
dps.sort()

df = pd.DataFrame(dps, columns=["valley_count", "seed", "filename"])
df.head()

,valley_count,seed,filename
0,10,a,../../experiment_data/multiseed_matching/a2_va...
1,10,b,../../experiment_data/multiseed_matching/a2_va...
2,10,c,../../experiment_data/multiseed_matching/a2_va...
3,25,a,../../experiment_data/multiseed_matching/a2_va...
4,25,b,../../experiment_data/multiseed_matching/a2_va...


In [36]:
def make_merged_feats(points, feats):
    in_feat = pd.merge(points, feats, on=["Feature ID", "Simplification Threshold"], how="left")
    in_feat.drop(columns=["Simplification Threshold", "Homogeneity", "Majority Class", "Major Class Size", "Majority Class Coverage"], inplace=True)
    in_feat.rename(columns={"Feature ID": "fid", "Data Index": "idx", "Loss Start": "lfrom", "Loss End": "lto", "Feature Type": "typ", "Volume": "vol", "Persistence": "pers"}, inplace=True)
    return in_feat

def make_between(a, b):
    between = pd.merge(a, b, on=["idx"], how="inner", suffixes=("_f", "_t"))
    intersection = between.groupby(["fid_f", "fid_t"]).size().reset_index(name="int")
    props = intersection.merge(between, on=["fid_f", "fid_t"], how="left")
    props["union"] = props["vol_f"] + props["vol_t"] - props["int"]
    props["iou"] = props["int"] / props["union"]

    return props

def get_best_iou_matching(between):
    best_iou = between.groupby("fid_f")["iou"].idxmax()
    best_matching = between.loc[best_iou]
    return best_matching

In [37]:
valleys = df["valley_count"].unique()

In [38]:
iou_ab = {}
iou_ac = {}
iou_bc = {}

iou_ab_min_avg = {}
iou_ac_min_avg = {}
iou_bc_min_avg = {}

for valley in valleys:
    points_a = pd.read_csv(df[(df["valley_count"] == valley) & (df["seed"] == "a")]["filename"].values[0].replace("feats", "points"))
    feats_a = pd.read_csv(df[(df["valley_count"] == valley) & (df["seed"] == "a")]["filename"].values[0])
    points_b = pd.read_csv(df[(df["valley_count"] == valley) & (df["seed"] == "b")]["filename"].values[0].replace("feats", "points"))
    feats_b = pd.read_csv(df[(df["valley_count"] == valley) & (df["seed"] == "b")]["filename"].values[0])
    points_c = pd.read_csv(df[(df["valley_count"] == valley) & (df["seed"] == "c")]["filename"].values[0].replace("feats", "points"))
    feats_c = pd.read_csv(df[(df["valley_count"] == valley) & (df["seed"] == "c")]["filename"].values[0])

    feat_a = make_merged_feats(points_a, feats_a)
    feat_b = make_merged_feats(points_b, feats_b)
    feat_c = make_merged_feats(points_c, feats_c)

    between_ab = make_between(feat_a, feat_b)
    between_ac = make_between(feat_a, feat_c)
    between_cb = make_between(feat_c, feat_b)

    between_ab.head()

    iou_ab[valley] = get_best_iou_matching(between_ab)
    iou_ac[valley] = get_best_iou_matching(between_ac)
    iou_bc[valley] = get_best_iou_matching(between_cb)

    iou_ab_min_avg[valley] = iou_ab[valley][iou_ab[valley]["typ_f"].str.contains("minima") & iou_ab[valley]["typ_t"].str.contains("minima")]["iou"].mean()
    iou_ac_min_avg[valley] = iou_ac[valley][iou_ac[valley]["typ_f"].str.contains("minima") & iou_ac[valley]["typ_t"].str.contains("minima")]["iou"].mean()
    iou_bc_min_avg[valley] = iou_bc[valley][iou_bc[valley]["typ_f"].str.contains("minima") & iou_bc[valley]["typ_t"].str.contains("minima")]["iou"].mean()

In [39]:
iou_ab_min_avg_df = pd.DataFrame(list(iou_ab_min_avg.items()), columns=["valley_count", "mean iou"])
iou_ac_min_avg_df = pd.DataFrame(list(iou_ac_min_avg.items()), columns=["valley_count", "mean iou"])
iou_bc_min_avg_df = pd.DataFrame(list(iou_bc_min_avg.items()), columns=["valley_count", "mean iou"])

iou_ab_min_avg_df["pair"] = "ab"
iou_ac_min_avg_df["pair"] = "ac"
iou_bc_min_avg_df["pair"] = "bc"

iou_min_avg_df = pd.concat([iou_ab_min_avg_df, iou_ac_min_avg_df, iou_bc_min_avg_df], ignore_index=True)

In [44]:
iou_ab[10][iou_ab[10]["typ_f"].str.contains("minima")]

,fid_f,fid_t,int,idx,lfrom_f,lto_f,typ_f,pers_f,vol_f,lfrom_t,lto_t,typ_t,pers_t,vol_t,union,iou
0,0,0,2067,1,-0.000000,0.205122,minima-saddle,0.000025,2934,-0.000000e+00,0.218696,minima-saddle,0.000026,15231,16098,0.128401
2934,1,0,7130,15,-0.000000,0.205122,minima-saddle,0.000023,16656,-0.000000e+00,0.218696,minima-saddle,0.000026,15231,24757,0.287999
19590,2,0,1044,124,-0.000000,0.205122,minima-saddle,0.000023,1398,-0.000000e+00,0.218696,minima-saddle,0.000026,15231,15585,0.066987
21730,4,5,560,135,-0.000000,0.205122,minima-saddle,0.000028,1410,-0.000000e+00,0.218696,minima-saddle,0.000056,1818,2668,0.209895
22665,5,10,1993,5,-0.000000,0.205122,minima-saddle,0.000048,2699,-0.000000e+00,0.218696,minima-saddle,0.000091,3711,4417,0.451211
25364,6,3,221,181,0.000001,0.205123,minima-saddle,0.000030,1094,7.152555e-07,0.218697,minima-saddle,0.000029,974,1847,0.119653
30069,8,16,1,24000,0.000016,0.205139,minima-saddle,0.000044,1,9.083335e-05,10.934902,saddle-maxima,10.934812,31346,31346,0.000032
69221,14,17,1,30221,0.000005,0.205128,minima-saddle,0.000026,3,3.075552e-05,0.218727,saddle-saddle,0.000031,1183,1185,0.000844
69222,15,16,1,56107,0.000044,0.205166,minima-saddle,0.000023,1,9.083335e-05,10.934902,saddle-maxima,10.934812,31346,31346,0.000032
69999,18,16,1,65354,0.000010,0.205132,minima-saddle,0.000027,1,9.083335e-05,10.934902,saddle-maxima,10.934812,31346,31346,0.000032


In [45]:
iou_ac[10][iou_ac[10]["typ_f"].str.contains("minima")]

,fid_f,fid_t,int,idx,lfrom_f,lto_f,typ_f,pers_f,vol_f,lfrom_t,lto_t,typ_t,pers_t,vol_t,union,iou
0,0,2,2342,1,-0.000000,0.205122,minima-saddle,0.000025,2934,-0.000000e+00,0.202480,minima-saddle,0.000063,3744,4336,0.540129
2934,1,0,5657,15,-0.000000,0.205122,minima-saddle,0.000023,16656,-0.000000e+00,0.202480,minima-saddle,0.000032,9910,20909,0.270553
20127,2,4,558,279,-0.000000,0.205122,minima-saddle,0.000023,1398,-0.000000e+00,0.202480,minima-saddle,0.000034,1584,2424,0.230198
21749,4,5,521,142,-0.000000,0.205122,minima-saddle,0.000028,1410,-0.000000e+00,0.202480,minima-saddle,0.000032,1462,2351,0.221608
22665,5,6,1932,5,-0.000000,0.205122,minima-saddle,0.000048,2699,-0.000000e+00,0.202480,minima-saddle,0.000103,3561,4328,0.446396
25364,6,3,245,261,0.000001,0.205123,minima-saddle,0.000030,1094,3.576278e-07,0.202480,minima-saddle,0.000024,987,1836,0.133442
30069,8,3,1,24000,0.000016,0.205139,minima-saddle,0.000044,1,3.576278e-07,0.202480,minima-saddle,0.000024,987,987,0.001013
69219,14,3,2,14243,0.000005,0.205128,minima-saddle,0.000026,3,3.576278e-07,0.202480,minima-saddle,0.000024,987,988,0.002024
69222,15,14,1,56107,0.000044,0.205166,minima-saddle,0.000023,1,4.446408e-05,0.202524,saddle-saddle,0.000022,839,839,0.001192
69999,18,15,1,65354,0.000010,0.205132,minima-saddle,0.000027,1,6.675498e-05,0.202547,saddle-saddle,0.000037,3625,3625,0.000276


In [46]:
iou_bc[10][iou_bc[10]["typ_f"].str.contains("minima")]

,fid_f,fid_t,int,idx,lfrom_f,lto_f,typ_f,pers_f,vol_f,lfrom_t,lto_t,typ_t,pers_t,vol_t,union,iou
0,0,0,5713,15,-0.000000e+00,0.202480,minima-saddle,0.000032,9910,-0.000000e+00,0.218696,minima-saddle,0.000026,15231,19428,0.294060
11959,1,1,2427,21,-0.000000e+00,0.202480,minima-saddle,0.000049,6950,-0.000000e+00,0.218696,minima-saddle,0.000026,3060,7583,0.320058
16860,2,0,2446,1,-0.000000e+00,0.202480,minima-saddle,0.000063,3744,-0.000000e+00,0.218696,minima-saddle,0.000026,15231,16529,0.147982
20604,3,3,225,225,3.576278e-07,0.202480,minima-saddle,0.000024,987,7.152555e-07,0.218697,minima-saddle,0.000029,974,1736,0.129608
21591,4,0,1124,26,-0.000000e+00,0.202480,minima-saddle,0.000034,1584,-0.000000e+00,0.218696,minima-saddle,0.000026,15231,15691,0.071633
23659,5,5,573,46,-0.000000e+00,0.202480,minima-saddle,0.000032,1462,-0.000000e+00,0.218696,minima-saddle,0.000056,1818,2707,0.211673
24637,6,10,2522,5,-0.000000e+00,0.202480,minima-saddle,0.000103,3561,-0.000000e+00,0.218696,minima-saddle,0.000091,3711,4750,0.530947
28198,7,6,1660,10,-0.000000e+00,0.202480,minima-saddle,0.000062,3239,-0.000000e+00,0.218696,minima-saddle,0.000028,2395,3974,0.417715
31487,9,8,30,2542,1.430510e-06,0.202481,minima-saddle,0.000023,166,1.072883e-06,0.218697,minima-saddle,0.000030,253,389,0.077121
68877,16,16,3,2647,1.978855e-05,0.202500,minima-saddle,0.000025,3,9.083335e-05,10.934902,saddle-maxima,10.934812,31346,31346,0.000096


In [40]:
iou_min_avg_df.sort_values(by=["valley_count", "pair"], inplace=True)
iou_min_avg_df

,valley_count,mean iou,pair
0,10,0.210691,ab
7,10,0.230670,ac
14,10,0.244533,bc
1,25,0.177353,ab
8,25,0.189736,ac
15,25,0.236505,bc
2,50,0.149024,ab
9,50,0.186439,ac
16,50,0.218211,bc
3,100,0.143338,ab


In [41]:
import plotly.express as px

px.line(iou_min_avg_df, x="valley_count", y="mean iou", color="pair", markers=True, title=f"Mean IoU of Minima Features Across Seeds for Layer {LAYER}")